# Time Series Classification Part 1: Feature Creation/Extraction

## B

In [2]:
import os
import glob
import pandas as pd

data_root = "AReM"
train_data = []
test_data = []

columns = ["time","avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]

# Iterate through the names of all files in the `data_root` directory.
for folder_name in os.listdir(data_root):
    folder_path = os.path.join(data_root, folder_name)
    if not os.path.isdir(folder_path):
        continue

    csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))# CSV file, sorted in ascending order

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        dataset_num = int(file_name.replace("dataset", "").replace(".csv", ""))
        rows = []

        with open(file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                # Remove commas at the end of lines
                line = line.rstrip(",")
                parts = line.split(",")
                if len(parts) == 7:
                    rows.append(parts)

        df = pd.DataFrame(rows, columns=columns)
        df = df.apply(pd.to_numeric)

        if folder_name in ["bending1", "bending2"]:
            if dataset_num in [1, 2]:
                test_data.append((folder_name, file_name, df))
            else:
                train_data.append((folder_name, file_name, df))
        else:
            if dataset_num in [1, 2, 3]:
                test_data.append((folder_name, file_name, df))
            else:
                train_data.append((folder_name, file_name, df))

print("Number of training set samples:", len(train_data))
print("Number of test set samples:", len(test_data))

Number of training set samples: 69
Number of test set samples: 19


## C

Commonly used time-domain features for time series classification include the minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, IQR, range. These features help describe a time series and are useful for classification.

In [19]:
import numpy as np
import pandas as pd


feature_rows = []
feature_names = ["Instance"]

for i in range(1, 7):
    feature_names.extend([
        f"min{i}",
        f"max{i}",
        f"mean{i}",
        f"median{i}",
        f"std{i}",
        f"1st quart{i}",
        f"3rd quart{i}"
    ])

instance = 1

for folder_name, file_name, df in all_data:
    features = [instance]
    sensor_data = df.drop(columns=["time"])

    for col in sensor_data.columns:
        features.extend([
            sensor_data[col].min(),
            sensor_data[col].max(),
            sensor_data[col].mean(),
            sensor_data[col].median(),
            sensor_data[col].std(),
            sensor_data[col].quantile(0.25),
            sensor_data[col].quantile(0.75)
        ])

    feature_rows.append(features)
    instance += 1

feature_df = pd.DataFrame(feature_rows, columns=feature_names)
feature_df = feature_df.set_index("Instance")

print(feature_df.shape)   
print(feature_df.head())

(88, 42)
           min1   max1      mean1  median1      std1  1st quart1  3rd quart1  \
Instance                                                                       
1         35.00  47.40  43.954500    44.33  1.558835       43.00       45.00   
2         33.00  47.75  42.179813    43.50  3.670666       39.15       45.00   
3         33.00  45.75  41.678063    41.75  2.243490       41.33       42.75   
4         37.00  48.00  43.454958    43.25  1.386098       42.50       45.00   
5         36.25  48.00  43.969125    44.50  1.618364       43.31       44.67   

          min2  max2     mean2  ...      std5  1st quart5  3rd quart5  min6  \
Instance                        ...                                           
1          0.0  1.70  0.426250  ...  1.999604     35.3625       36.50   0.0   
2          0.0  3.00  0.696042  ...  3.849448     30.4575       36.33   0.0   
3          0.0  2.83  0.535979  ...  2.411026     28.4575       31.25   0.0   
4          0.0  1.58  0.378083  ...

In [22]:
X = feature_df
feature_std = X.std()
n_boot = 1000
result = []

for col in X.columns:
    boot_std = []
    for _ in range(n_boot):
        sample = X[col].sample(n=len(X), replace=True)
        boot_std.append(sample.std())

    lower = np.percentile(boot_std, 5)
    upper = np.percentile(boot_std, 95)

    result.append([col, feature_std[col], lower, upper])

ci_df = pd.DataFrame(result, columns=["Feature", "Standard Deviation", "90% CI Lower", "90% CI Upper"])

print(ci_df)

       Feature  Standard Deviation  90% CI Lower  90% CI Upper
0         min1            9.624011      8.306206     10.758194
1         max1            4.207745      3.183435      5.140538
2        mean1            5.276431      4.598078      5.844423
3      median1            5.386624      4.734175      5.974569
4         std1            1.771282      1.580727      1.944725
5   1st quart1            6.127846      5.504877      6.592375
6   3rd quart1            5.031028      4.238230      5.751037
7         min2            0.000000      0.000000      0.000000
8         max2            5.059656      4.633505      5.390823
9        mean2            1.577908      1.396834      1.699224
10     median2            1.413545      1.245273      1.545724
11        std2            0.885875      0.802228      0.939158
12  1st quart2            0.948434      0.836148      1.035548
13  3rd quart2            2.131337      1.884847      2.300562
14        min3            2.954516      2.762977      3

Based on the extracted features, I choose mean, standard deviation, and maximum because they describe the average value, the variability, and the peak value of the signal. These three features capture different aspects of the time series and are likely to be more informative for classification.

In [25]:
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

# 保存每个训练样本的特征
feature_rows = []

instance = 1

for folder_name, file_name, df in train_data:

    # 设置标签
    if folder_name in ["bending1", "bending2"]:
        label = "bending"
    else:
        label = "other"

    features = [instance, label]

    # 去掉 time 列
    sensor_data = df.drop(columns=["time"])

    # 提取每个时间序列的统计特征
    for col in sensor_data.columns:
        features.append(sensor_data[col].min())
        features.append(sensor_data[col].max())
        features.append(sensor_data[col].mean())
        features.append(sensor_data[col].median())
        features.append(sensor_data[col].std())
        features.append(sensor_data[col].quantile(0.25))
        features.append(sensor_data[col].quantile(0.75))

    feature_rows.append(features)
    instance += 1

# 列名
feature_names = ["Instance", "Label"]

for i in range(1, 7):
    feature_names.extend([
        f"min{i}",
        f"max{i}",
        f"mean{i}",
        f"median{i}",
        f"std{i}",
        f"Q1_{i}",
        f"Q3_{i}"
    ])

feature_df = pd.DataFrame(feature_rows, columns=feature_names)

print(feature_df.head())

   Instance    Label   min1   max1      mean1  median1      std1   Q1_1  \
0         1  bending  35.00  47.40  43.954500    44.33  1.558835  43.00   
1         2  bending  33.00  47.75  42.179813    43.50  3.670666  39.15   
2         3  bending  33.00  45.75  41.678063    41.75  2.243490  41.33   
3         4  bending  37.00  48.00  43.454958    43.25  1.386098  42.50   
4         5  bending  36.25  48.00  43.969125    44.50  1.618364  43.31   

    Q3_1  min2  ...      std5     Q1_5   Q3_5  min6  max6     mean6  median6  \
0  45.00   0.0  ...  1.999604  35.3625  36.50   0.0  1.79  0.493292     0.43   
1  45.00   0.0  ...  3.849448  30.4575  36.33   0.0  2.18  0.613521     0.50   
2  42.75   0.0  ...  2.411026  28.4575  31.25   0.0  1.79  0.383292     0.43   
3  45.00   0.0  ...  2.488862  22.2500  24.00   0.0  5.26  0.679646     0.50   
4  44.67   0.0  ...  3.318301  20.5000  23.75   0.0  2.96  0.555313     0.49   

       std6  Q1_6  Q3_6  
0  0.513506  0.00  0.94  
1  0.524317  0.0